# Лабораторная 6. Когда данных много и как они доходят до модели

К этому моменту система работает: данные приезжают, проверяются, складываются
в модель, собираются в витрины, всё запускается по расписанию и переживает
перезапуск. На ста тысячах заказов она отрабатывает за секунды.

А потом бизнес вырастает в сто раз, и по очереди ломаются четыре вещи. Чтение
файла перестаёт помещаться в память. Полный пересчёт витрины идёт четыре часа,
а окно – сорок минут. Антифроду нужен ответ за доли секунды, а не «завтра
в отчёте». И наконец, человек, обучающий модель, просит «те же данные, что были
в момент заказа», – а у нас в таблице лежит дата доставки, то есть будущее.

Каждая из четырёх поломок решается по-своему, и ни одна – покупкой модного
инструмента. Разбираться будем по порядку.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **Партиционирование – partitioning** | Разбиение данных на части по значению колонки, чаще всего по дате, чтобы читать только нужное. |
| **Распределённая обработка – distributed processing** | Счёт на нескольких машинах: данные делят на части, считают отдельно, результаты сводят. |
| **Событие – event** | Запись о том, что произошло: заказ оплачен, кнопка нажата. Событие не изменяют, добавляют новое. |
| **Журнал событий – topic** | Упорядоченная лента событий, из которой несколько потребителей читают со своей скоростью. |
| **Смещение – offset** | Позиция потребителя в журнале: место, до которого он дочитал. |
| **Доставка «хотя бы один раз» – at-least-once** | Гарантия, при которой событие не потеряется, но может прийти повторно. |
| **Таблица признаков – feature table** | Данные в том виде, в каком их принимает модель: строка-объект, колонки-признаки, колонка-ответ. |
| **Утечка ответа – feature leakage** | Ситуация, когда в признаки попало то, что станет известно только после предсказываемого события. |

## Подготовка

Следующая ячейка подключит компактные учебные аналоги хранилища и брокера
событий. Кластеры Spark или Kafka для этой работы не нужны.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import time

import duckdb
import numpy as np
import pandas as pd
import paths
from course_visuals import show_external_visual, show_visual
from lab_utils import require_artifact
from minikafka import Consumer, Topic, produce_events
from stages import ensure_stage

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# ЛР 6 использует факт, который был построен в ЛР 4. При независимом запуске
# (например, в новой Colab-сессии) восстанавливаем только предыдущий CORE.
ensure_stage("core")
require_artifact(paths.CLEAN / "clean_orders.parquet", "ЛР 3")
con = duckdb.connect(str(paths.WAREHOUSE))

In [ ]:
# Пересобираем факт: ноутбук не должен зависеть от состояния чужого ядра.
for t in ["clean_orders", "clean_items", "clean_payments", "clean_reviews",
          "clean_customers", "clean_products", "clean_sellers"]:
    con.execute(f"""CREATE OR REPLACE TABLE {t} AS SELECT * FROM
                    read_parquet('{(paths.CLEAN / (t + ".parquet")).as_posix()}')""")

# Явная зависимость от ЛР 4: таблица фактов создаётся там.
if not con.execute("""SELECT COUNT(*) FROM duckdb_tables()
                      WHERE schema_name = 'core' AND table_name = 'fact_order_items'
                   """).fetchone()[0]:
    raise RuntimeError(
        "В хранилище нет таблицы core.fact_order_items.\n"
        "Она создаётся в ЛР 4 (задание 1). Выполните ЛР 4 полностью "
        "и запустите этот ноутбук ещё раз.")

n_fact = con.execute("SELECT COUNT(*) FROM core.fact_order_items").fetchone()[0]
print(f"строк в core.fact_order_items: {n_fact:,}")

## Когда одной машины перестаёт хватать

pandas считает на одной машине и исходит из того, что нужные данные целиком
помещаются в память. Упирается это обычно не в размер файла: на каждом шаге
появляются промежуточные копии, текстовые колонки в памяти весят заметно
больше, чем на диске, а если рядом работает коллега, память делится на двоих.

Порог «после стольких-то гигабайт нужен кластер» назвать нельзя: всё зависит
от того, что именно считается, за какое время должен прийти ответ, сколько
запросов идёт одновременно и сколько стоит содержать решение. Разумный путь –
идти по возрастанию сложности и останавливаться, как только хватило:

| Шаг | Что делаем | Когда достаточно |
|-----|-----------|------------------|
| 1 | читаем только нужные колонки и строки (Parquet + фильтр) | почти всегда помогает первым |
| 2 | обрабатываем порциями (chunks) | агрегаты, независимые по строкам |
| 3 | считаем в СУБД, а не в Python (DuckDB, ClickHouse, PostgreSQL) | всё, что выражается в SQL |
| 4 | **вертикальное масштабирование**: машина побольше | одной машине уже тесно, но задача проста |
| 5 | **горизонтальное масштабирование**: много машин (Spark) | нужны несколько машин или много параллельных вычислений |

> **Вертикальное масштабирование** – сделать одну машину мощнее.
> Просто, но упирается в потолок и стоит нелинейно дорого.
>
> **Горизонтальное масштабирование** – добавить машин и распределить работу.
> Потолка почти нет, но появляются сеть, отказы узлов и сложность отладки.

Первые три варианта закрывают большинство задач «у нас большие данные» и
экономят бюджет. Настоящих терабайтных задач в обычной компании меньше,
чем кажется.

In [ ]:
# Сравним три способа посчитать одно и то же на самом большом нашем файле.
big_csv = paths.RAW_OLIST / "olist_geolocation_dataset.csv"
big_pq = paths.PARQUET / "olist_geolocation_dataset.parquet"

t0 = time.perf_counter()
full = pd.read_csv(big_csv)
mem_mb = full.memory_usage(deep=True).sum() / 1024 ** 2
res_full = full.groupby("geolocation_state").size()
t_full = time.perf_counter() - t0
del full

t0 = time.perf_counter()
partial = {}
for chunk in pd.read_csv(big_csv, usecols=["geolocation_state"], chunksize=200_000):
    for state, cnt in chunk.groupby("geolocation_state").size().items():
        partial[state] = partial.get(state, 0) + cnt
res_chunk = pd.Series(partial).sort_index()
t_chunk = time.perf_counter() - t0

t0 = time.perf_counter()
res_sql = con.execute(f"""
    SELECT geolocation_state, COUNT(*) AS n
    FROM read_parquet('{big_pq.as_posix()}')
    GROUP BY 1 ORDER BY 1
""").df()
t_sql = time.perf_counter() - t0

print(f"файл на диске (CSV)           : {big_csv.stat().st_size / 1024**2:6.1f} МБ")
print(f"он же в памяти pandas         : {mem_mb:6.1f} МБ")
print(f"отношение память/диск         : {mem_mb / (big_csv.stat().st_size / 1024**2):6.2f}×")
print()
print(f"1) прочитать целиком в pandas : {t_full:5.2f} c")
print(f"2) порциями по 200 тыс. строк : {t_chunk:5.2f} c  (в памяти – одна порция)")
print(f"3) SQL поверх Parquet         : {t_sql:5.2f} c")
print(f"\nрезультаты совпадают: "
      f"{res_full.sort_index().tolist() == res_chunk.tolist() == res_sql['n'].tolist()}")

Отношение «память / диск» здесь около единицы: свежие версии pandas хранят
строки эффективно, а колонки у нас в основном числовые. Но это не повод расслабляться – на текстовых
колонках в старом `object`-представлении то же самое отношение легко
оказывается 3–5×, а сверху добавляются временные копии, которые pandas
создаёт на каждом преобразовании. Правило «у меня 8 ГБ оперативки, значит
обработаю 8 ГБ данных» не работает: нужен кратный запас.

Работают здесь способы 2 и 3. Обработка порциями не требует никаких новых
технологий, это просто цикл. А перенос расчёта в SQL-движок дал на наших
данных выигрыш на порядок, потому что движок читает только нужную колонку
и не строит python-объектов. Огромное количество задач «у нас большие данные»
закрывается этими двумя приёмами, а не кластером.

## Как не читать лишнее

> **Партиционирование** – физическое разделение данных на части по значению
> колонки (чаще всего по дате), так что запрос с фильтром по этой колонке
> читает только нужные части.

```
  data/marts/fact_partitioned/
  ├── year_month=2017-01/data_0.parquet
  ├── year_month=2017-02/data_0.parquet
  ├── …
  └── year_month=2018-08/data_0.parquet

  WHERE year_month = '2018-03'  →  движок читает ОДИН файл вместо двадцати
```

Это называется **partition pruning** (отсечение партиций) и даёт, пожалуй,
лучшее соотношение «эффект / усилие» во всей инженерии данных.

Как выбирать ключ партиционирования:

* по тому, по чему **чаще всего фильтруют** (обычно это дата);
* так, чтобы партиции получались **не слишком мелкими**: тысячи файлов
  по 200 КБ работают медленнее, чем десятки по 50 МБ;
* так, чтобы данные можно было **перезаписывать целой партицией** – это
  и есть идемпотентность из пятой работы, только на уровне файлов.

Плохие ключи: `customer_id` (миллионы крошечных партиций), случайный хэш
(фильтровать по нему никто не будет).

## Как делят работу между машинами

Идея распределённой обработки умещается в одну фразу: разбей данные на части,
посчитай частичные результаты независимо, сложи их вместе.

In [ ]:
show_visual("spark")

Кто именно раздаёт работу и кто её выполняет – видно на схеме из документации
Spark:

In [ ]:
show_external_visual(
    "spark_cluster",
    caption="Driver раздаёт задачи, executors на worker nodes их выполняют.")

Для нас здесь важны две роли. **Driver Program** – управляющий процесс: он
решает, на какие части разбить работу и какие задачи запустить. **Executor** –
исполнитель на рабочей машине, он и обрабатывает свой кусок данных; `Task`
внутри него – одна такая порция работы. Cluster Manager раздаёт машины
и для этой работы нам не нужен.

Spark – движок, который делает ровно это, но всерьёз: распределяет
части по машинам кластера, переносит вычисление к данным, переживает падение
узла (пересчитает потерянную часть), умеет перемешивать данные между узлами
(shuffle – самая дорогая операция) и оптимизирует план запроса.

Вот эта же идея на нашем ноутбуке, без всякого кластера:

In [ ]:
# «Spark своими руками»: считаем выручку по заказам, обрабатывая части по одной.
items_pq = paths.PARQUET / "olist_order_items_dataset.parquet"
data = pd.read_parquet(items_pq, columns=["order_id", "price"])


def map_partition(df: pd.DataFrame) -> pd.DataFrame:
    """MAP: частичный агрегат по одной части данных."""
    return df.groupby("order_id", as_index=False)["price"].sum()


def reduce_partials(parts: list[pd.DataFrame]) -> pd.DataFrame:
    """REDUCE: объединяем частичные результаты."""
    return pd.concat(parts).groupby("order_id", as_index=False)["price"].sum()


N_PARTS = 8
chunks = [data.iloc[i::N_PARTS] for i in range(N_PARTS)]   # имитируем 8 партиций
partials = [map_partition(c) for c in chunks]              # ← это можно делать параллельно
combined = reduce_partials(partials)
direct = map_partition(data)

print(f"частей обработано: {len(partials)}, макс. размер части: {max(map(len, chunks)):,} строк")
print(f"заказов в результате: {len(combined):,} (прямой расчёт: {len(direct):,})")
print(f"суммы совпали: {abs(combined['price'].sum() - direct['price'].sum()) < 0.01}")
print("\nЭто и есть модель вычислений Spark. Всё остальное в нём – про то,")
print("как делать это на 200 машинах и что делать, когда три из них упали.")

Тот же расчёт на PySpark выглядел бы так (запускать не нужно – важно узнать
конструкции, когда встретите их в чужом коде):

```python
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("olist").getOrCreate()

items = spark.read.parquet("s3://bucket/staging/order_items/")
orders = spark.read.parquet("s3://bucket/staging/orders/")

revenue = (items.join(orders, "order_id")                 # ← shuffle, дорого
                .withColumn("month", F.date_trunc("month", "order_purchase_timestamp"))
                .groupBy("month")
                .agg(F.sum("price").alias("revenue")))

revenue.write.mode("overwrite").partitionBy("month").parquet("s3://bucket/marts/revenue/")
```

**Когда Spark нужен:** данные не помещаются на одну машину (сотни гигабайт
и терабайты), обработка неструктурированных данных пачками, тяжёлые
преобразования, которые не выражаются в SQL.

**Когда Spark не нужен:** объём в гигабайтах; задача выражается одним SQL;
в команде нет людей, готовых эксплуатировать кластер. Spark на 10 ГБ данных
обычно **медленнее** DuckDB на одной машине – за счёт накладных расходов
на координацию.

## Задание 1. Партиционирование факта

Сохраните `core.fact_order_items` в партиционированном виде:

```
  data/marts/fact_partitioned/year_month=YYYY-MM/…parquet
```

Требования:

1. Ключ партиционирования – колонка `year_month` формата `YYYY-MM`,
   вычисленная из `order_date`.
2. Число партиций равно числу месяцев в данных.
3. Суммарное число строк во всех партициях равно числу строк факта.
4. Внутри партиции лежат **только** строки своего месяца.

Подсказка по инструменту (не по решению): в DuckDB запись с партиционированием
делается через `COPY ... TO '<папка>' (FORMAT PARQUET, PARTITION_BY (<колонка>),
OVERWRITE_OR_IGNORE)`. Подумайте, где взять `year_month` и почему перезапись
папки здесь так же важна, как в пятой работе.

In [ ]:
PART_DIR = paths.MARTS / "fact_partitioned"

sql_partition = f"""
-- TODO: ваш запрос
SELECT 1
"""

con.execute(sql_partition)
parts = sorted(p.name for p in PART_DIR.glob("year_month=*"))
print(f"партиций создано: {len(parts)}")
print(parts[:5], "…")

In [ ]:
# Проверим, что фильтр по партиции действительно читает меньше данных.
if not list(PART_DIR.rglob("*.parquet")):
    raise SystemExit("Партиции ещё не созданы – сначала выполните задание выше.")

all_bytes = sum(f.stat().st_size for f in PART_DIR.rglob("*.parquet"))
one_bytes = sum(f.stat().st_size
                for f in (PART_DIR / "year_month=2018-03").rglob("*.parquet"))

t0 = time.perf_counter()
one_month = con.execute(f"""
    SELECT COUNT(*) AS строк, ROUND(SUM(price), 2) AS выручка
    FROM read_parquet('{(PART_DIR / "**" / "*.parquet").as_posix()}', hive_partitioning = 1)
    WHERE year_month = '2018-03'
""").df()
t_part = time.perf_counter() - t0

t0 = time.perf_counter()
all_data = con.execute(f"""
    SELECT COUNT(*) AS строк, ROUND(SUM(price), 2) AS выручка
    FROM read_parquet('{(PART_DIR / "**" / "*.parquet").as_posix()}', hive_partitioning = 1)
""").df()
t_all = time.perf_counter() - t0

print(f"весь партиционированный набор : {all_bytes / 1024**2:7.2f} МБ")
print(f"партиция year_month=2018-03   : {one_bytes / 1024**2:7.2f} МБ "
      f"({one_bytes / all_bytes:.1%} от объёма)")
print()
print(f"запрос за март   : {t_part * 1000:6.1f} мс → {one_month.to_dict('records')}")
print(f"запрос по всему  : {t_all * 1000:6.1f} мс → {all_data.to_dict('records')}")
print("""
Главное здесь – не миллисекунды (на таком объёме разница тонет в шуме), а проценты:
запрос за март физически трогает несколько процентов данных. Когда набор
вырастет до сотен гигабайт, эти проценты превратятся в разницу между
«секунда» и «полчаса», причём без единой новой технологии.
""")

In [ ]:
from checks.lab06 import (check_task_1, check_task_2, check_task_3, lab06_result)

check_task_1(con, PART_DIR)

## Когда ждать до утра уже нельзя

Пакетная обработка отвечает на вопрос «что было вчера». Для отчётности этого
достаточно, но есть задачи, где вчерашний ответ бесполезен: остановить
подозрительный платёж, показать «пять человек смотрят этот товар», предупредить
о сбое на линии.

Первое, что приходит в голову, – писать события прямо в базу и читать оттуда.
Это работает, пока потребитель один. Как только их становится трое – антифрод,
аналитика, уведомления, – начинаются проблемы: у каждого своя скорость, своё
«с какого места читать», и все они конкурируют за одну таблицу. Плюс база
хранит текущее состояние, а не историю событий: восстановить, что именно
происходило вчера в 14:05, по ней уже не получится.

Так появляется отдельный компонент – брокер событий. Самый известный
представитель – Apache Kafka, и на его словаре удобно разобрать устройство.

Данные лежат в **журнале (topic)**: упорядоченная лента записей, которые
добавляются в конец и не меняются. Читатель не удаляет событие – он двигает
своё **смещение (offset)**, то есть отметку «до какого места я дочитал».
Поэтому три потребителя спокойно читают одну ленту со своей скоростью, а
упавший может вернуться назад и перечитать пропущенное.

Тот, кто пишет события, называется **производителем (producer)**, кто
читает – **потребителем (consumer)**.

In [ ]:
show_visual("stream")

Две вещи, без которых картина неполная.

**Разделы (partitions).** Один журнал физически разбит на несколько частей.
Событие попадает в раздел по ключу – например, по номеру карты, – и внутри
раздела порядок сохраняется. Разделы дают две вещи сразу: можно читать их
параллельно разными процессами и можно быть уверенным, что все события одной
карты обработает один потребитель и в правильном порядке.

**Группа потребителей (consumer group).** Если один процесс не справляется,
запускают несколько с одним именем группы – и брокер сам раздаёт им разделы,
по одному разделу на процесс. Это горизонтальное масштабирование чтения.
А если нужно, чтобы те же события независимо прочитала другая система,
ей дают другое имя группы, и она получает всю ленту целиком.

На схеме из документации Kafka это выглядит так:

In [ ]:
show_external_visual(
    "kafka_consumer_groups",
    caption="Разделы P0–P3 поделены между потребителями двух независимых групп.")

`P0`–`P3` – разделы одного топика, лежащие на двух серверах. `C1`–`C6` –
потребители. Группа A состоит из двух процессов, и четыре раздела поделены
между ними пополам; группа B – из четырёх, и каждому достался свой раздел.
Обе группы читают одни и те же события независимо друг от друга, каждая
со своей скоростью.

Отсюда практическое следствие: число разделов задаёт потолок параллельности.
Четыре раздела – не больше четырёх одновременно читающих процессов в группе,
пятый будет простаивать.

### Гарантии доставки – и откуда берутся дубликаты

| Гарантия | Что значит | Цена |
|----------|------------|------|
| at-most-once | не больше одного раза, возможна потеря | быстро, но теряем данные |
| **at-least-once** | не меньше одного раза, возможны дубликаты | offset фиксируем после обработки |
| exactly-once | эффект обработки один раз в оговорённой границе | нужны транзакции или идемпотентный sink |

Семантика – свойство всего пути, а не одного брокера. Зафиксировали offset до
записи результата – возможна потеря; после записи – возможна повторная
доставка. Kafka поддерживает транзакционные сценарии внутри Kafka, но внешняя
БД или API всё равно требуют транзакцию, дедупликационный ключ либо
идемпотентный обработчик. Повтор события – нормальная инженерная ситуация;
это ровно та же идея, что и в пятой работе, только на уровне одного сообщения.

In [ ]:
# Заводим поток событий об оплате заказов.
events_df = con.execute("""
    SELECT o.order_id,
           'pay_' || o.order_id            AS event_id,
           p.payment_type,
           p.payment_value                 AS amount
    FROM clean_orders o
    JOIN (SELECT order_id, min_by(payment_type, payment_sequential) AS payment_type,
                 SUM(payment_value) AS payment_value
          FROM clean_payments GROUP BY 1) p USING (order_id)
    ORDER BY o.order_purchase_timestamp
    LIMIT 5000
""").df()

topic = Topic("payments")
produce_events(topic, events_df.to_dict("records"), duplicate_rate=0.08)

unique_events = events_df["event_id"].nunique()
print(f"уникальных событий у источника : {unique_events:,}")
print(f"записей попало в топик         : {len(topic):,}  ← источник продублировал часть")
print(f"ожидаемая сумма (по уникальным): {events_df['amount'].sum():,.2f}")

In [ ]:
# Наивный обработчик: складывает всё, что прочитал.
naive_state = {"n_events": 0, "total_amount": 0.0}
consumer = Consumer(topic, group="naive")
while True:
    batch = consumer.poll(500)
    if not batch:
        break
    for rec in batch:
        naive_state["n_events"] += 1
        naive_state["total_amount"] += rec.value["amount"]
    consumer.commit()

print(f"наивный обработчик насчитал   : {naive_state['n_events']:,} событий, "
      f"сумма {naive_state['total_amount']:,.2f}")
print(f"а должно быть                 : {unique_events:,} событий, "
      f"сумма {events_df['amount'].sum():,.2f}")
print(f"завышение суммы               : "
      f"{naive_state['total_amount'] / events_df['amount'].sum() - 1:.2%}")

## Задание 2. Идемпотентный обработчик потока

Напишите функцию `apply_event(state, record)`, которая обновляет состояние
так, чтобы **повторная обработка того же события ничего не меняла**.

Структура состояния:

```python
state = {
    "seen": set(),          # идентификаторы уже обработанных событий
    "n_events": 0,          # число уникальных обработанных событий
    "total_amount": 0.0,    # сумма по уникальным событиям
    "by_type": {},          # {payment_type: сумма по уникальным событиям}
}
```

Требования:

1. После обработки всего топика `n_events` равно числу **уникальных**
   `event_id`, а `total_amount` – сумме по уникальным событиям.
2. Обработчик должен пережить **падение до коммита**: consumer перечитает
   часть записей заново, и это не должно изменить результат.
3. Повторный прогон всего топика с нуля не меняет состояние.

Подумайте: что здесь играет роль ключа идемпотентности и почему `order_id`
для этого не годится, если у заказа может быть несколько платежей.

In [ ]:
def new_state() -> dict:
    return {"seen": set(), "n_events": 0, "total_amount": 0.0, "by_type": {}}


def apply_event(state: dict, record) -> dict:
    """Идемпотентно применяет одно событие к состоянию."""
    # TODO: сделайте применение события идемпотентным
    e = record.value
    state["n_events"] += 1
    state["total_amount"] += e["amount"]
    state["by_type"][e["payment_type"]] = \
        state["by_type"].get(e["payment_type"], 0.0) + e["amount"]
    return state


def consume_all(topic: Topic, state: dict, crash_at: int | None = None) -> dict:
    """Читает топик до конца; при crash_at имитирует падение до коммита."""
    consumer = Consumer(topic, group="idempotent")
    crashed = False
    while True:
        batch = consumer.poll(500)
        if not batch:
            break
        for rec in batch:
            apply_event(state, rec)
        if crash_at is not None and not crashed and consumer.position >= crash_at:
            consumer.crash()         # откат к последнему коммиту → перечитаем заново
            crashed = True
            continue
        consumer.commit()
    return state


state = consume_all(topic, new_state(), crash_at=1500)
print(f"обработано уникальных событий : {state['n_events']:,}")
print(f"сумма                          : {state['total_amount']:,.2f}")
print(f"по типам оплаты                : "
      f"{ {k: round(v, 2) for k, v in sorted(state['by_type'].items())} }")

In [ ]:
check_task_2(state, events_df, topic)

### Как выбирать между пакетной и потоковой обработкой

| Вопрос | Пакетная | Потоковая |
|--------|---------------|-------------------|
| Какая допустимая задержка? | часы | секунды |
| Что сломается, если опоздать? | ничего страшного | потеряем деньги / не остановим мошенника |
| Нужна ли история для расчёта? | да, обычно | обычно только окно |
| Сколько стоит эксплуатация? | низкая | дежурства, мониторинг лага, состояние |

Потоковая обработка не заменяет пакетную. В одной системе живут оба
контура: поток считает то, что нужно сейчас, а пакетный раз в сутки
пересчитывает историю с учётом опоздавших событий. **Lambda Architecture** – более узкий,
классический подход: он сознательно разделяет batch layer и speed layer, а
затем объединяет их результаты в serving layer. Не всякая система, где есть
и пакетный, и потоковый контур, является lambda-архитектурой.

## Что модель принимает на вход

Модель – ещё один потребитель данных, только требовательный.

In [ ]:
show_visual("features")

> **Feature (признак)** – числовая или категориальная характеристика объекта,
> которую модель использует для предсказания.
>
> **Feature pipeline** – воспроизводимый процесс, превращающий данные
> хранилища в таблицу признаков с зафиксированным определением каждого признака.
>
> **Обучающая выборка (training dataset)** – таблица «объект → признаки → ответ».

И главная ловушка, из-за которой модели, прекрасные на тестах, разваливаются
в продакшене:

> **Утечка ответа (feature leakage)** – попадание в признаки информации,
> которой **не существовало в момент, когда предсказание должно быть сделано**.

У нас задача: в момент оформления заказа предсказать, опоздает ли доставка.
Значит, доступны: состав заказа, цена, доставка, штаты покупателя и продавца,
обещанный срок. И **недоступны**: фактическая дата доставки, фактический срок,
отзыв покупателя, финальный статус заказа. Их ещё не существует.

Признак `delivery_days` – это не «сильный признак». Это переписанный ответ.

### Почему сплит по времени, а не случайный

Случайный сплит перемешивает прошлое и будущее: модель обучается на данных
июня и проверяется на данных марта. В реальности так не бывает – модель
всегда работает «в будущее». Сплит по времени воспроизводит именно эту
ситуацию и потому честнее, даже если метрика получается ниже.

Стандартный набор – три части: **train** (учим), **validation** (подбираем
параметры), **test** (один раз в конце смотрим честную оценку). Мы для
компактности используем две.

## Задание 3. Таблица признаков без утечки

Соберите `features` – **одна строка на доставленный заказ** – со следующими
колонками:

| колонка | определение |
|---------|-------------|
| `order_id` | ключ |
| `order_date` | дата покупки (для сплита, не признак) |
| `n_items` | число позиций в заказе |
| `items_total` | сумма `price` |
| `freight_total` | сумма `freight_value` |
| `n_sellers` | число различных продавцов в заказе |
| `n_categories` | число различных категорий |
| `total_weight_g` | суммарный вес товаров (пропуски считать нулём) |
| `seller_state` | штат продавца с минимальным `seller_id` в заказе |
| `customer_state` | штат покупателя |
| `same_state` | продавец и покупатель в одном штате |
| `estimated_days` | сколько дней обещали на доставку (от покупки до `order_estimated_delivery_date`) |
| `purchase_month`, `purchase_dow` | месяц и день недели покупки |
| `payment_total` | сумма платежей по заказу (0, если платежей нет) |
| `installments` | максимальное число частей рассрочки (1, если платежей нет) |
| `payment_type` | тип первой платёжной операции по `payment_sequential` (`'unknown'`, если платежей нет) |
| `is_late` | **ответ модели**: доставка позже обещанного (0/1) |

Требования:

1. Только доставленные заказы – у остальных ответа ещё не существует.
2. Одна строка на заказ. Позиции нужно агрегировать, а не джойнить «как есть»:
   иначе заказ из пяти позиций попадёт в обучение пять раз и перевесит
   остальные.
3. **Ни одного признака, которого не существует в момент оформления заказа.**
   Проверка ищет такие колонки по именам и по поведению модели.
4. Без пропусков в признаках.

У заказа может быть несколько продавцов и несколько платёжных операций. Для
этой учебной витрины правило выбора намеренно фиксировано: берём продавца с
минимальным `seller_id` и первую платёжную операцию по `payment_sequential`.
Это не универсальная бизнес-логика, а способ сделать один заказ одной
воспроизводимой строкой. В реальном проекте такое правило согласуют с
владельцем продукта или строят отдельную витрину другого зерна.

Источники: `core.fact_order_items`, `core.dim_product`, `core.dim_seller`,
`core.dim_customer`, `clean_orders`, `clean_payments`.

In [ ]:
sql_features = """
-- TODO: ваш запрос
SELECT 1
"""

features = con.execute(sql_features).df()
print(f"строк: {len(features):,}, колонок: {features.shape[1]}")
if "is_late" in features.columns:
    print(f"доля опоздавших: {features['is_late'].mean():.3%}")
features.head(3)

### Сплит по времени и обучение

Дальше – код преподавателя: он один и тот же для всех, чтобы сравнение было
честным. Ваша часть работы закончилась на таблице признаков – и это ровно
та граница, где Data Engineering передаёт данные в ML.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SPLIT_DATE = pd.Timestamp("2018-06-01")
CAT_FEATURES = ["seller_state", "customer_state", "payment_type", "same_state"]
NUM_FEATURES = ["n_items", "items_total", "freight_total", "n_sellers",
                "n_categories", "total_weight_g", "estimated_days",
                "purchase_month", "purchase_dow", "payment_total", "installments"]


def train_and_score(df: pd.DataFrame, num_features: list[str]) -> dict:
    missing = [c for c in ["order_date", "is_late", *num_features, *CAT_FEATURES]
               if c not in df.columns]
    if missing:
        raise KeyError(f"в таблице признаков не хватает колонок {missing} – "
                       "сначала выполните задание 3")
    df = df.copy()
    df["order_date"] = pd.to_datetime(df["order_date"])
    train = df[df["order_date"] < SPLIT_DATE]
    valid = df[df["order_date"] >= SPLIT_DATE]
    cols = num_features + CAT_FEATURES
    model = SkPipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20),
             CAT_FEATURES)])),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    model.fit(train[cols], train["is_late"])
    proba = model.predict_proba(valid[cols])[:, 1]
    return {"n_train": len(train), "n_valid": len(valid),
            "late_rate_train": float(train["is_late"].mean()),
            "late_rate_valid": float(valid["is_late"].mean()),
            "auc": float(roc_auc_score(valid["is_late"], proba))}


honest = train_and_score(features, NUM_FEATURES)
print("Обучение на честных признаках:")
for k, v in honest.items():
    print(f"  {k:<16}: {v:.4f}" if isinstance(v, float) else f"  {k:<16}: {v:,}")

In [ ]:
check_task_3(features, honest, con)

## Что ломается в этом месте

**Добавили признак «сколько дней шла доставка».**

In [ ]:
leaky = con.execute("""
    SELECT o.order_id, CAST(o.order_purchase_timestamp AS DATE) AS order_date,
           o.delivery_days
    FROM clean_orders o WHERE o.is_delivered
""").df()
leaky_features = features.merge(leaky[["order_id", "delivery_days"]], on="order_id")
leaked = train_and_score(leaky_features, NUM_FEATURES + ["delivery_days"])

print(f"AUC на честных признаках : {honest['auc']:.4f}")
print(f"AUC с delivery_days      : {leaked['auc']:.4f}   ← «отличная модель»")
print("""
Метрика почти единица – и это не успех, а диагноз. Признак delivery_days
вычисляется из даты доставки, то есть из того самого события, которое мы
предсказываем. В продакшене в момент оформления заказа его просто нет:
модель либо упадёт, либо получит пропуск и начнёт выдавать мусор.

Правило, по которому ловят утечку: если метрика внезапно стала подозрительно
хорошей – сначала ищите утечку, и только потом радуйтесь.
""")

**Не привели данные к зерну заказа.**

In [ ]:
item_grain = con.execute("""
    SELECT f.order_id, f.order_date, CAST(f.is_late AS INTEGER) AS is_late
    FROM core.fact_order_items f WHERE f.is_delivered
""").df()
print(f"строк на grain заказа   : {len(features):,},  доля опоздавших: "
      f"{features['is_late'].mean():.3%}")
print(f"строк на grain позиции  : {len(item_grain):,},  доля опоздавших: "
      f"{item_grain['is_late'].mean():.3%}")
print("""
Таргет тот же, а его доля другая: заказы с большим числом позиций опаздывают
чаще и при обучении «на позициях» получают больший вес. Модель научится
немного не тому, а виновата будет не она.
""")

**Разделили выборку случайно, а не по времени.**

In [ ]:
from sklearn.model_selection import train_test_split

rnd_train, rnd_valid = train_test_split(features, test_size=0.2, random_state=0)
cols = NUM_FEATURES + CAT_FEATURES
rnd_model = SkPipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_FEATURES)])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
]).fit(rnd_train[cols], rnd_train["is_late"])
rnd_auc = roc_auc_score(rnd_valid["is_late"],
                        rnd_model.predict_proba(rnd_valid[cols])[:, 1])

print(f"AUC при сплите по времени : {honest['auc']:.4f}")
print(f"AUC при случайном сплите  : {rnd_auc:.4f}")
print(f"доля опоздавших: train {honest['late_rate_train']:.3%}, "
      f"valid {honest['late_rate_valid']:.3%}")
print("""
Числа могут оказаться близкими – и это нормально: случайный сплит не всегда
завышает метрику. Важно другое: он отвечает на другой вопрос. Сплит по времени
показывает, как модель поведёт себя на данных, которых она не видела, – включая
изменение самой доли опоздавших между периодами. Так будет и
в продакшене.
""")

## Где заканчивается Data Engineering и начинается MLOps

> **Feature Store** – система, которая хранит признаки с их определениями
> и отдаёт их двум потребителям сразу: обучению (историчные значения
> на момент события) и продакшену (текущие значения с малой задержкой).

Проблема, которую он решает, называется **training/serving skew**: признак
«средний чек клиента за 30 дней» посчитан в обучении одним SQL, а в проде –
другим кодом на другом языке. Расхождение в пару процентов – и модель в проде
работает хуже, чем на тестах, а причину ищут месяцами.

Где проходит граница ответственности:

```
  Data Engineering            │  ML / MLOps
  источники, качество,        │  выбор модели, обучение, метрики,
  модель данных, витрины,     │  деплой, мониторинг качества модели,
  feature pipeline,           │  переобучение, A/B-тесты
  воспроизводимость данных    │
```

Общая часть – feature pipeline: он одновременно последний шаг инженера
и первый шаг ML. Поэтому DE и ML-инженеры обычно и делят его на двоих.

И вывод всего курса одной фразой:

> Модель не может быть лучше данных, на которых она обучена. Всё, что мы
> делали пять работ – схемы, проверки качества, зерно таблиц, идемпотентность,
> воспроизводимость – это и есть то, что отделяет работающую ML-систему
> от демонстрации на ноутбуке.

In [ ]:
# Сохраняем таблицу признаков – это финальный артефакт сквозного проекта.
features_path = paths.MARTS / "features_orders.parquet"
features.to_parquet(features_path, index=False)
print("сохранено:", features_path.relative_to(paths.ROOT))

pd.DataFrame([honest]).assign(модель="honest").to_parquet(
    paths.MARTS / "model_metrics.parquet", index=False)
print("метрики модели сохранены")

## Та же задача в другой системе

Платёжный сервис обрабатывает около 3000 транзакций в секунду в пике.
Требования:

* **блокировать подозрительную транзакцию до её подтверждения** – на решение
  есть 200 мс;
* аналитикам – ежедневный отчёт по мошенничеству и по потерям;
* модели антифрода – переобучение раз в неделю на исторических данных
  за 2 года;
* расследование инцидента: по любой транзакции нужно уметь показать,
  какие признаки видела модель в момент решения.

Ответьте письменно:

1. Какие здесь источники и какие два контура обработки?
2. Где нужна потоковая обработка, где пакетная и где они пересекаются?
3. Как признак «число транзакций этой карты за последний час» считается
   в онлайне и как – при обучении? В чём риск расхождения?
4. Нужен ли здесь Kafka? А Spark? Обоснуйте оба ответа.
5. Что делать с дубликатами событий и почему at-least-once здесь особенно опасен?
6. Как обеспечить требование «показать, что видела модель в момент решения»?
7. Что в этой архитектуре будет лишним?

### Ваш ответ

1. Источники и контуры: …
2. Потоковая и пакетная обработка: …
3. Онлайн- и офлайн-признаки: …
4. Kafka и Spark: …
5. Дубликаты: …
6. Воспроизводимость решения модели: …
7. Лишнее: …

In [ ]:
lab06_result()
con.close()

## Вопросы для проверки себя

1. У вас 30 МБ данных в сутки и расчёт, который укладывается в минуту.
   Нужен ли Spark? Что должно измениться, чтобы ответ стал другим?
2. Партиционирование по `customer_id` – почему это плохая идея,
   даже если по клиенту часто фильтруют?
3. Почему топик в Kafka – это не очередь? Какое свойство журнала
   со смещениями это даёт и зачем оно нужно на практике?
4. Модель показала AUC 0,98 на валидации и 0,61 в продакшене. Назовите три
   правдоподобные причины, начиная с самой вероятной.
5. Почему сплит по времени честнее случайного, если метрика при нём обычно
   хуже? В каком случае случайный сплит допустим?

## Финал курса

Мы прошли путь от девяти CSV до воспроизводимой цепочки: сырьё, проверенный
слой, модель данных, витрины, признаки, обученная модель. Кластер для этого
не понадобился – и это нормальное инженерное решение, пока объём и требования
к задержке не просят большего.

Чего мы не делали и о чём теперь знаете: настоящий Airflow с расписанием
и дежурствами, кластер Kafka с репликацией, Spark на терабайтах, dbt на сотне
моделей, таблицы Iceberg в объектном хранилище, хранилище признаков. Вы
знаете, какую задачу решает каждая из этих вещей, – этого достаточно, чтобы
разобраться с конкретным инструментом за несколько дней, когда он понадобится.

Полезнее всего из курса – не список технологий, а привычка спрашивать: что
означает одна строка, что произойдёт при повторном запуске, какая допустимая
задержка и что сломается, если её превысить. Эти вопросы остаются одними
и теми же независимо от того, что написано в вакансии.

Разбор архитектурных ситуаций из других отраслей – в папке
`architecture_cases`: десять коротких описаний систем, где те же вопросы
приводят к разным ответам.